# Linear Algebra from Scratch

**Mathematics for Machine Learning from First Principles · Lesson 2**

A dataset is a matrix whose rows are examples and whose columns are features. A linear model multiplies that matrix by a vector of weights, a neural-network layer multiplies by a weight matrix, and nearest-neighbour methods compare vectors by their distance. This notebook builds those operations from plain Python lists, so that every number in a matrix product is one you have computed yourself.

## Learning objectives

By the end of this notebook you will be able to:

1. represent scalars, vectors and matrices with Python numbers and lists, and validate their shapes;
2. implement vector addition, subtraction, scalar multiplication and the dot product;
3. implement matrix multiplication with the classic triple loop, rejecting empty, ragged and incompatible inputs with a clear `ValueError`;
4. compute transposes, identity matrices, the $L^1$, $L^2$ and $L^\infty$ norms, and distances;
5. transform 2-D points by scaling, rotation and shear, and show that multiplying matrices composes those transformations.

## Dependencies

**Tier 0 — Python standard library only.** The core cells import only `math`, `random`, `copy` and `numbers`. Two optional cells compare the results with NumPy and draw them with Matplotlib. Each is labelled *Optional*, tagged `comparison` or `visualization`, checks that its library is installed before using it, and is never used by a later cell.

The notebook runs in JupyterLite (Python 3.14 inside your browser) and on your JupyterHub server (Python 3.13). It needs no internet access and runs from top to bottom in a few seconds.

### How to use this notebook

Run the cells from top to bottom. Each implementation cell is followed by a test cell of plain `assert` statements. The exercises use the same `check` helper as Lesson 1, defined in the next cell: it prints `not implemented yet` until you write an exercise, prints a pass line when your function is right, and raises `AssertionError` naming the failing call when it is wrong. Reference solutions are at the end.

In [ ]:
import random

import copy
import math
import numbers


def _matches(result, expected, rel_tol, abs_tol):
    """True when result equals expected: exactly for integers and booleans,
    within a tolerance for floats, and entry by entry for lists and tuples."""
    if expected is None or isinstance(expected, bool):
        return result is expected
    if isinstance(expected, numbers.Real):
        if isinstance(result, bool) or not isinstance(result, numbers.Real):
            return False
        if isinstance(expected, int):
            return result == expected
        return math.isclose(result, expected, rel_tol=rel_tol, abs_tol=abs_tol)
    if isinstance(expected, (list, tuple)):
        return (
            isinstance(result, (list, tuple))
            and len(result) == len(expected)
            and all(_matches(r, e, rel_tol, abs_tol) for r, e in zip(result, expected))
        )
    return result == expected


def check(name, function, cases, *, rel_tol=1e-9, abs_tol=1e-9, strict=False):
    """Test an exercise function against a list of (arguments, expected) cases.

    arguments is a tuple of positional arguments, for example ([1, 2, 3],) for a
    function that takes one list. expected is the value the function should
    return, or an exception class such as ValueError that it should raise.
    Integers must match exactly; floats are compared with math.isclose.

    Prints "<name>: not implemented yet" while the function still raises
    NotImplementedError, prints a pass line when every case is right, and raises
    AssertionError with the failing call when a case is wrong. strict=True
    (used in the Solutions section) treats an unimplemented function as a failure.
    """
    for arguments, expected in cases:
        call = f"{name}({', '.join(repr(argument) for argument in arguments)})"
        wants_error = isinstance(expected, type) and issubclass(expected, Exception)
        # Work on copies of list and dict arguments so that a function which
        # changes its input is caught instead of silently corrupting later cases.
        supplied = tuple(
            copy.deepcopy(argument) if isinstance(argument, (list, dict)) else argument
            for argument in arguments
        )
        try:
            result = function(*supplied)
        except NotImplementedError:
            if strict:
                raise AssertionError(f"{call} raised NotImplementedError; finish the function first")
            print(f"{name}: not implemented yet")
            return
        except Exception as error:
            if wants_error and isinstance(error, expected):
                continue
            wanted = f"it to raise {expected.__name__}" if wants_error else repr(expected)
            raise AssertionError(
                f"{call} raised {type(error).__name__}: {error}; expected {wanted}"
            ) from error
        if wants_error:
            raise AssertionError(f"{call} returned {result!r}; expected it to raise {expected.__name__}")
        for before, after in zip(arguments, supplied):
            if isinstance(before, (list, dict)) and before != after:
                raise AssertionError(
                    f"{call} changed its input {before!r} into {after!r}; "
                    "return a new list instead of modifying the argument"
                )
        if not _matches(result, expected, rel_tol, abs_tol):
            raise AssertionError(f"{call} returned {result!r}; expected {expected!r}")
    summary = "the test" if len(cases) == 1 else f"all {len(cases)} tests"
    print(f"{name}: {summary} passed")



def raises_value_error(function, *arguments):
    """True when function(*arguments) raises ValueError (used by the tests below)."""
    try:
        function(*arguments)
    except ValueError:
        return True
    return False

## Theory

### Scalars, vectors and matrices

A **scalar** is a single number, $c \in \mathbb{R}$. A **vector** $\mathbf{x} \in \mathbb{R}^n$ is an ordered list of $n$ numbers, and a **matrix** $A \in \mathbb{R}^{m \times n}$ is a rectangular array with $m$ rows and $n$ columns:

$$\mathbf{x} = \begin{pmatrix} x_1 \\ \vdots \\ x_n \end{pmatrix}, \qquad A = \begin{pmatrix} a_{11} & \cdots & a_{1n} \\ \vdots & \ddots & \vdots \\ a_{m1} & \cdots & a_{mn} \end{pmatrix}.$$

Here a vector is a Python list and a matrix is a list of rows, so $a_{ij}$ is `A[i - 1][j - 1]`. A dataset with $m$ examples and $n$ features is an $m \times n$ matrix whose $i$-th row is the feature vector of example $i$.

Mathematically, the dot product of two empty vectors is the empty sum, $0$. In machine-learning code, however, an empty vector or matrix almost always means that something upstream went wrong, such as a filter that removed every row. The functions in this notebook therefore reject empty input with a `ValueError` instead of quietly returning $0$.

### Vector operations

Vectors of the same length combine entry by entry, and a scalar scales every entry:

$$\mathbf{x} + \mathbf{y} = (x_1 + y_1, \ldots, x_n + y_n), \qquad \mathbf{x} - \mathbf{y} = (x_1 - y_1, \ldots, x_n - y_n), \qquad c\,\mathbf{x} = (c\,x_1, \ldots, c\,x_n).$$

The **dot product** multiplies matching entries and adds the results:

$$\mathbf{x} \cdot \mathbf{y} = \sum_{i=1}^{n} x_i\, y_i = \|\mathbf{x}\|_2\, \|\mathbf{y}\|_2 \cos\theta,$$

where $\theta$ is the angle between the two vectors. A dot product of $0$ means the vectors are perpendicular, or **orthogonal**. The score of a linear model, $\mathbf{w} \cdot \mathbf{x} + b$, is a dot product plus a bias.

### Matrix operations

Matrices of the same shape add entry by entry. The **product** of $A \in \mathbb{R}^{m \times n}$ and $B \in \mathbb{R}^{n \times p}$ is the $m \times p$ matrix

$$(AB)_{ij} = \sum_{k=1}^{n} a_{ik}\, b_{kj},$$

so entry $(i, j)$ is the dot product of row $i$ of $A$ with column $j$ of $B$. The **inner** sizes must agree (the $n$ in $m \times n$ and $n \times p$), and the **outer** sizes give the shape of the result. Three nested loops compute it: one over the rows $i$, one over the columns $j$, and one over the shared index $k$.

Matrix multiplication is **associative**, $(AB)C = A(BC)$, but it is **not commutative**: $AB$ and $BA$ usually differ, and one of them may not even be defined. The **transpose** swaps rows and columns, $(A^\top)_{ij} = a_{ji}$, and it reverses the order of a product: $(AB)^\top = B^\top A^\top$. The **identity** matrix $I_n$ has ones on its diagonal and zeros elsewhere, and leaves every matrix unchanged: $I_m A = A I_n = A$ for $A \in \mathbb{R}^{m \times n}$. A matrix times a vector, $A\mathbf{x}$, is the special case in which $\mathbf{x}$ is a single column.

### Norms and distances

A **norm** measures the size of a vector. Three are common in machine learning:

$$\|\mathbf{x}\|_1 = \sum_{i=1}^{n} |x_i|, \qquad \|\mathbf{x}\|_2 = \sqrt{\sum_{i=1}^{n} x_i^2}, \qquad \|\mathbf{x}\|_\infty = \max_{i} |x_i|.$$

The **distance** between two vectors is the norm of their difference, $d(\mathbf{x}, \mathbf{y}) = \|\mathbf{x} - \mathbf{y}\|$. The $L^2$ (Euclidean) distance drives k-nearest neighbours and k-means clustering; the $L^1$ and $L^2$ norms of a weight vector are the lasso and ridge penalties; and the $L^\infty$ norm bounds the largest single entry. Every norm satisfies the **triangle inequality** $\|\mathbf{x} + \mathbf{y}\| \le \|\mathbf{x}\| + \|\mathbf{y}\|$, and every vector satisfies $\|\mathbf{x}\|_\infty \le \|\mathbf{x}\|_2 \le \|\mathbf{x}\|_1$.

### Linear transformations of the plane

A $2 \times 2$ matrix $M$ moves each point $\mathbf{p} = (x, y)$ of the plane to $M\mathbf{p}$. The first column of $M$ is where $(1, 0)$ lands, and the second column is where $(0, 1)$ lands. Three building blocks are

$$S(s_x, s_y) = \begin{pmatrix} s_x & 0 \\ 0 & s_y \end{pmatrix}, \qquad R(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}, \qquad H(k) = \begin{pmatrix} 1 & k \\ 0 & 1 \end{pmatrix}.$$

$S$ scales each axis, $R(\theta)$ rotates anticlockwise by $\theta$ radians, and the shear $H(k)$ slides each point sideways in proportion to its height: $(x, y) \mapsto (x + ky,\ y)$.

Applying $B$ first and then $A$ gives $A(B\mathbf{p}) = (AB)\mathbf{p}$, so the single matrix $AB$ performs both steps, **the rightmost one first**. Because matrix multiplication is not commutative, the order of the steps matters.

## Step-by-step implementation

### Step 1: shapes and validation

Every later function relies on two checks: that a vector is not empty, and that a matrix is a non-empty rectangle. `all_close` compares numbers, vectors or matrices with a small tolerance, for results that pass through `math.cos` and `math.sin`.

In [ ]:
def check_vector(x, name="vector"):
    """Return len(x), raising ValueError if x is empty."""
    if len(x) == 0:
        raise ValueError(f"the {name} is empty")
    return len(x)


def check_same_length(x, y):
    """Return the common length of x and y, raising ValueError if the lengths differ."""
    n = check_vector(x, "first vector")
    m = check_vector(y, "second vector")
    if n != m:
        raise ValueError(f"the vectors must have the same length, got {n} and {m}")
    return n


def shape(matrix):
    """Return (rows, columns) of a list-of-rows matrix; reject empty or ragged input."""
    if len(matrix) == 0:
        raise ValueError("a matrix needs at least one row")
    columns = len(matrix[0])
    if columns == 0:
        raise ValueError("a matrix needs at least one column")
    for i, row in enumerate(matrix):
        if len(row) != columns:
            raise ValueError(f"row {i} has {len(row)} entries but row 0 has {columns}: the rows are ragged")
    return len(matrix), columns


def all_close(a, b, abs_tol=1e-12):
    """True when two numbers, vectors or matrices agree entry by entry within abs_tol."""
    if isinstance(a, list) and isinstance(b, list):
        return len(a) == len(b) and all(all_close(x, y, abs_tol) for x, y in zip(a, b))
    return math.isclose(a, b, rel_tol=1e-12, abs_tol=abs_tol)

In [ ]:
# Tests for Step 1
assert shape([[1, 2, 3], [4, 5, 6]]) == (2, 3), "a 2 x 3 matrix has 2 rows and 3 columns"
assert shape([[1], [2], [3]]) == (3, 1), "a column vector has one column"
assert shape([[1, 2, 3]]) == (1, 3), "a row vector has one row"
assert raises_value_error(shape, []), "an empty list is not a matrix"
assert raises_value_error(shape, [[]]), "a matrix with an empty row has no columns"
assert raises_value_error(shape, [[1, 2], [3]]), "ragged rows do not form a matrix"
assert check_same_length([1, 2], [3, 4]) == 2, "two vectors of length 2"
assert raises_value_error(check_same_length, [1, 2], [1, 2, 3]), "lengths 2 and 3 do not match"
assert raises_value_error(check_vector, []), "an empty vector is rejected"
assert all_close([[1.0, 0.0]], [[1.0, 1e-17]]), "all_close ignores differences far below the tolerance"
assert not all_close([1.0, 2.0], [1.0, 2.1]), "all_close notices a real difference"
assert not all_close([1.0, 2.0], [1.0, 2.0, 3.0]), "all_close notices a missing entry"
print("Step 1 tests passed")

**Expected behaviour.** Shapes are reported as `(rows, columns)`. An empty list, an empty row and ragged rows each raise `ValueError` with a message that says which rule was broken. `all_close` treats $10^{-17}$ as zero but notices a difference of $0.1$ and a missing entry.

### Step 2: vector operations

Each function validates its inputs first and then applies the formula from the theory. `zip` pairs up matching entries; because `check_same_length` has already confirmed that the lengths agree, `zip` cannot silently drop any entries.

In [ ]:
def vector_add(x, y):
    """x + y, entry by entry."""
    check_same_length(x, y)
    return [xi + yi for xi, yi in zip(x, y)]


def vector_subtract(x, y):
    """x - y, entry by entry."""
    check_same_length(x, y)
    return [xi - yi for xi, yi in zip(x, y)]


def scalar_multiply(c, x):
    """c x: multiply every entry of x by the scalar c."""
    check_vector(x)
    return [c * xi for xi in x]


def dot(x, y):
    """The dot product: the sum of x_i * y_i."""
    check_same_length(x, y)
    total = 0
    for xi, yi in zip(x, y):
        total += xi * yi
    return total

In [ ]:
# Tests for Step 2
x, y = [1, 2, 3], [4, 5, 6]
assert vector_add(x, y) == [5, 7, 9], "[1, 2, 3] + [4, 5, 6] = [5, 7, 9]"
assert vector_subtract(x, y) == [-3, -3, -3], "[1, 2, 3] - [4, 5, 6] = [-3, -3, -3]"
assert all_close(scalar_multiply(2, [1, -2, 0.5]), [2, -4, 1.0]), "2 [1, -2, 0.5] = [2, -4, 1.0]"
assert dot(x, y) == 32, "1*4 + 2*5 + 3*6 = 32"
assert dot([1, 2], [-2, 1]) == 0, "perpendicular vectors have a dot product of 0"
assert dot(x, y) == dot(y, x), "the dot product is symmetric"
z = [-1, 0, 2]
assert dot(x, vector_add(y, z)) == dot(x, y) + dot(x, z), "the dot product distributes over addition"
assert raises_value_error(vector_add, [1, 2], [1, 2, 3]), "vectors of different lengths cannot be added"
assert raises_value_error(dot, [], []), "empty vectors are rejected"
assert x == [1, 2, 3] and y == [4, 5, 6], "the operations must not modify their inputs"
print("Step 2 tests passed: x . y =", dot(x, y))
print("For comparison, Python's own + on lists concatenates:", [1, 2, 3] + [4, 5, 6])
print("and * on a list repeats it:", 2 * [1, 2, 3])

**Expected behaviour.** All the tests pass and the dot product prints as `32`. The last two lines show why these functions are needed: for lists, Python's `+` means concatenation and `*` means repetition, not vector arithmetic.

### Step 3: matrix operations

`matrix_multiply` is the classic triple loop. Read its loops against the formula $(AB)_{ij} = \sum_k a_{ik}\, b_{kj}$: `i` chooses a row of $A$, `j` chooses a column of $B$, and `k` walks along both at once. The function validates both shapes before it computes anything, and its error message states the shapes involved.

In [ ]:
def matrix_add(A, B):
    """A + B for two matrices of the same shape."""
    if shape(A) != shape(B):
        raise ValueError(f"cannot add a {shape(A)} matrix to a {shape(B)} matrix")
    return [[a + b for a, b in zip(row_a, row_b)] for row_a, row_b in zip(A, B)]


def matrix_scalar_multiply(c, A):
    """c A: multiply every entry of A by the scalar c."""
    shape(A)
    return [[c * a for a in row] for row in A]


def transpose(A):
    """The n x m matrix whose entry (i, j) is A[j][i]."""
    m, n = shape(A)
    return [[A[i][j] for i in range(m)] for j in range(n)]


def identity(n):
    """The n x n identity matrix."""
    if n < 1:
        raise ValueError(f"an identity matrix needs n >= 1, got {n}")
    return [[1 if i == j else 0 for j in range(n)] for i in range(n)]


def matrix_multiply(A, B):
    """The product AB, computed with the classic triple loop.

    A must be m x n and B must be n x p; the result is m x p. Raises ValueError
    for an empty or ragged matrix and when the inner sizes differ.
    """
    m, n = shape(A)
    n_b, p = shape(B)
    if n != n_b:
        raise ValueError(
            f"cannot multiply a {m}x{n} matrix by a {n_b}x{p} matrix: "
            f"A has {n} columns but B has {n_b} rows"
        )
    C = [[0] * p for _ in range(m)]  # a new list for every row
    for i in range(m):  # each row of A
        for j in range(p):  # each column of B
            total = 0
            for k in range(n):  # along row i of A and down column j of B
                total += A[i][k] * B[k][j]
            C[i][j] = total
    return C


def matrix_vector_multiply(A, x):
    """A x: entry i is the dot product of row i of A with x."""
    m, n = shape(A)
    if len(x) != n:
        raise ValueError(f"a {m}x{n} matrix needs a vector of length {n}, got {len(x)}")
    return [dot(row, x) for row in A]

In [ ]:
# Tests for Step 3: hand-calculated products
A = [[1, 2, 3],
     [4, 5, 6]]  # 2 x 3
B = [[7, 8],
     [9, 10],
     [11, 12]]  # 3 x 2
assert matrix_multiply(A, B) == [[58, 64], [139, 154]], "(2 x 3)(3 x 2): for example 1*7 + 2*9 + 3*11 = 58"
assert matrix_multiply(B, A) == [[39, 54, 69], [49, 68, 87], [59, 82, 105]], "(3 x 2)(2 x 3) is a 3 x 3 matrix"
row = [[1, 2, 3]]  # 1 x 3
col = [[4], [5], [6]]  # 3 x 1
assert matrix_multiply(row, col) == [[32]], "a row times a column is a 1 x 1 matrix holding the dot product"
assert matrix_multiply(col, row) == [[4, 8, 12], [5, 10, 15], [6, 12, 18]], "a column times a row is a 3 x 3 matrix"
assert matrix_vector_multiply(A, [1, 0, -1]) == [-2, -2], "[1 - 3, 4 - 6] = [-2, -2]"
assert transpose(A) == [[1, 4], [2, 5], [3, 6]], "the transpose of the 2 x 3 matrix A is 3 x 2"
assert identity(3) == [[1, 0, 0], [0, 1, 0], [0, 0, 1]], "the 3 x 3 identity"
assert matrix_add(A, A) == matrix_scalar_multiply(2, A) == [[2, 4, 6], [8, 10, 12]], "A + A = 2A"
print("AB =", matrix_multiply(A, B))
print("BA =", matrix_multiply(B, A))

In [ ]:
# Tests for Step 3: invalid shapes and the rules of matrix algebra
for left, right in [
    (A, A),  # (2 x 3)(2 x 3): A has 3 columns but the right-hand A has 2 rows
    (A, [[1, 2], [3, 4]]),  # (2 x 3)(2 x 2)
    ([], B),  # no rows at all
    ([[]], B),  # a row with no columns
    ([[1, 2], [3]], B),  # ragged left-hand side
    (B, [[1, 2, 3], [4, 5]]),  # ragged right-hand side
]:
    assert raises_value_error(matrix_multiply, left, right), f"matrix_multiply({left}, {right}) should raise ValueError"
assert raises_value_error(matrix_vector_multiply, A, [1, 2]), "a 2 x 3 matrix needs a vector of length 3"
assert raises_value_error(matrix_add, A, B), "matrices of different shapes cannot be added"
assert raises_value_error(identity, 0), "an identity matrix needs at least one row"
try:
    matrix_multiply(A, A)
except ValueError as error:
    print("ValueError:", error)

C = [[1, -1], [2, 0]]
assert matrix_multiply(matrix_multiply(A, B), C) == matrix_multiply(A, matrix_multiply(B, C)), "(AB)C = A(BC)"
assert matrix_multiply(identity(2), A) == A == matrix_multiply(A, identity(3)), "I_2 A = A = A I_3"
assert transpose(matrix_multiply(A, B)) == matrix_multiply(transpose(B), transpose(A)), "(AB)^T = B^T A^T"
P, Q = [[1, 2], [3, 4]], [[0, 1], [1, 0]]
assert matrix_multiply(P, Q) != matrix_multiply(Q, P), "PQ and QP differ: multiplication is not commutative"

rng = random.Random(0)
for _ in range(50):
    m, n, p = rng.randint(1, 4), rng.randint(1, 4), rng.randint(1, 4)
    M = [[rng.randint(-9, 9) for _ in range(n)] for _ in range(m)]
    N = [[rng.randint(-9, 9) for _ in range(p)] for _ in range(n)]
    product = matrix_multiply(M, N)
    assert shape(product) == (m, p), "an (m x n)(n x p) product has shape m x p"
    for j in range(p):
        column_j = [N[k][j] for k in range(n)]
        assert [r[j] for r in product] == matrix_vector_multiply(M, column_j), "column j of MN must be M times column j of N"
print("Step 3 tests passed")

**Expected behaviour.** The products match the hand calculations. For example, entry $(1, 1)$ of $AB$ is $1 \cdot 7 + 2 \cdot 9 + 3 \cdot 11 = 58$, and swapping the order gives a $3 \times 3$ matrix instead of a $2 \times 2$ one. Every invalid input raises `ValueError`, and the printed message explains the mismatch: `cannot multiply a 2x3 matrix by a 2x3 matrix: A has 3 columns but B has 2 rows`. The algebraic rules (associativity, the identity and the transpose of a product) hold exactly, because these tests use integers. Fifty random integer products have the right shapes and agree, column by column, with `matrix_vector_multiply`.

### Step 4: norms and distances

Each norm is one line of Python. `distance` takes the norm as an argument, so the same function measures Euclidean, Manhattan or Chebyshev distance.

In [ ]:
def norm_l1(x):
    """The L1 (Manhattan) norm: the sum of absolute values."""
    check_vector(x)
    return sum(abs(xi) for xi in x)


def norm_l2(x):
    """The L2 (Euclidean) norm: the square root of the sum of squares."""
    check_vector(x)
    return math.sqrt(sum(xi * xi for xi in x))


def norm_linf(x):
    """The L-infinity (maximum) norm: the largest absolute value."""
    check_vector(x)
    return max(abs(xi) for xi in x)


def distance(x, y, norm=norm_l2):
    """The distance between x and y measured with the given norm (L2 by default)."""
    return norm(vector_subtract(x, y))

In [ ]:
# Tests for Step 4
v = [3, -4]
assert norm_l1(v) == 7, "|3| + |-4| = 7"
assert math.isclose(norm_l2(v), 5.0), "sqrt(9 + 16) = 5"
assert norm_linf(v) == 4, "max(|3|, |-4|) = 4"
p, q = [1, 2], [4, 6]
assert math.isclose(distance(p, q), 5.0), "the Euclidean distance from (1, 2) to (4, 6) is 5"
assert distance(p, q, norm_l1) == 7, "the Manhattan distance is 3 + 4 = 7"
assert distance(p, q, norm_linf) == 4, "the Chebyshev distance is max(3, 4) = 4"
assert math.isclose(distance(p, p), 0.0, abs_tol=1e-12), "a point is at distance 0 from itself"
assert raises_value_error(norm_l2, []), "the norm of an empty vector is rejected"
assert raises_value_error(distance, [1, 2], [1, 2, 3]), "distance needs vectors of the same length"

rng = random.Random(1)
for _ in range(200):
    n = rng.randint(1, 6)
    a = [rng.uniform(-10, 10) for _ in range(n)]
    b = [rng.uniform(-10, 10) for _ in range(n)]
    c = rng.uniform(-3, 3)
    assert norm_linf(a) <= norm_l2(a) * (1 + 1e-12), "expected the L-infinity norm <= the L2 norm"
    assert norm_l2(a) <= norm_l1(a) * (1 + 1e-12), "expected the L2 norm <= the L1 norm"
    assert norm_l2(vector_add(a, b)) <= (norm_l2(a) + norm_l2(b)) * (1 + 1e-12), "the triangle inequality failed"
    assert math.isclose(norm_l2(scalar_multiply(c, a)), abs(c) * norm_l2(a), rel_tol=1e-12, abs_tol=1e-12), "||c a|| should equal |c| ||a||"
    assert math.isclose(distance(a, b), math.dist(a, b), rel_tol=1e-12), "distance should agree with math.dist"
print("Step 4 tests passed")

huge = [1e200, 1e200]
print("norm_l2([1e200, 1e200]) =", norm_l2(huge), "but math.hypot(1e200, 1e200) =", math.hypot(*huge))

**Expected behaviour.** For $\mathbf{v} = (3, -4)$ the three norms are $7$, $5$ and $4$, and the distances from $(1, 2)$ to $(4, 6)$ are $5$, $7$ and $4$. On 200 random vectors the norms keep their order, satisfy the triangle inequality and scale with $|c|$, and `distance` agrees with `math.dist`. The last line shows a limitation: `norm_l2` returns `inf` for `[1e200, 1e200]`, because squaring $10^{200}$ overflows, whereas `math.hypot` rescales internally and returns about $1.41 \times 10^{200}$.

### Step 5: linear transformations of 2-D points

Each transformation is a $2 \times 2$ matrix, and `transform_points` applies one to a list of points with `matrix_vector_multiply`. The tests check hand-calculated corners of the unit square, the special properties of rotations, and the central fact that a matrix product composes transformations.

In [ ]:
def scaling_matrix(sx, sy):
    """S(sx, sy): stretch x by sx and y by sy."""
    return [[sx, 0], [0, sy]]


def rotation_matrix(theta):
    """R(theta): rotate anticlockwise by theta radians (use math.radians for degrees)."""
    c, s = math.cos(theta), math.sin(theta)
    return [[c, -s], [s, c]]


def shear_matrix(k):
    """H(k): the horizontal shear (x, y) -> (x + k y, y)."""
    return [[1, k], [0, 1]]


def transform_points(M, points):
    """Apply the 2 x 2 matrix M to every point [x, y] in a list of points."""
    if shape(M) != (2, 2):
        raise ValueError(f"expected a 2 x 2 matrix, got shape {shape(M)}")
    return [matrix_vector_multiply(M, point) for point in points]

In [ ]:
# Tests for Step 5
square = [[0, 0], [1, 0], [1, 1], [0, 1]]  # corners of the unit square
assert transform_points(scaling_matrix(2, 3), square) == [[0, 0], [2, 0], [2, 3], [0, 3]], "S(2, 3) stretches x by 2 and y by 3"
assert all_close(transform_points(shear_matrix(0.5), square), [[0, 0], [1, 0], [1.5, 1], [0.5, 1]]), "H(0.5) slides each point right by half its height"
quarter_turn = rotation_matrix(math.pi / 2)
assert all_close(transform_points(quarter_turn, [[1, 0], [0, 1]]), [[0, 1], [-1, 0]]), "a quarter turn takes (1, 0) to (0, 1) and (0, 1) to (-1, 0)"
assert all_close(quarter_turn, [[0, -1], [1, 0]]), "R(90 degrees) is [[0, -1], [1, 0]] up to rounding"
assert quarter_turn[0][0] != 0, "cos(pi / 2) comes out as a tiny number, not exactly 0, because pi / 2 is rounded"
assert raises_value_error(transform_points, [[1, 2, 3], [4, 5, 6]], square), "only 2 x 2 matrices transform points of the plane"

rng = random.Random(2)
for _ in range(100):
    point = [rng.uniform(-5, 5), rng.uniform(-5, 5)]
    theta = rng.uniform(-math.pi, math.pi)
    turned = matrix_vector_multiply(rotation_matrix(theta), point)
    assert math.isclose(norm_l2(turned), norm_l2(point), rel_tol=1e-12, abs_tol=1e-12), "a rotation must preserve length"

S, R = scaling_matrix(2, 1), rotation_matrix(math.pi / 2)
start = [1, 1]
rotate_then_scale = matrix_vector_multiply(S, matrix_vector_multiply(R, start))
assert all_close(matrix_vector_multiply(matrix_multiply(S, R), start), rotate_then_scale), "(SR)p = S(Rp): the product applies R first"
assert all_close(rotate_then_scale, [-2, 1]), "rotate (1, 1) to (-1, 1), then double x: (-2, 1)"
scale_then_rotate = matrix_vector_multiply(matrix_multiply(R, S), start)
assert all_close(scale_then_rotate, [-1, 2]), "double x to get (2, 1), then rotate: (-1, 2)"
a, b = 0.3, 1.1
assert all_close(matrix_multiply(rotation_matrix(a), rotation_matrix(b)), rotation_matrix(a + b)), "R(a)R(b) = R(a + b)"
assert all_close(matrix_multiply(shear_matrix(0.5), shear_matrix(0.25)), shear_matrix(0.75)), "shears add: H(0.5)H(0.25) = H(0.75)"
print("Step 5 tests passed")

**Expected behaviour.** The scaled and sheared squares match the hand-calculated corners exactly. Rotation results agree only within a tiny tolerance, because the float `math.pi / 2` is not exactly $\pi/2$, so its cosine comes out at about $6 \times 10^{-17}$ rather than $0$. Rotations preserve lengths, rotating by $a$ and then by $b$ equals rotating by $a + b$, and shears add. Most importantly, $(SR)\mathbf{p} = S(R\mathbf{p})$: one matrix product does the work of two steps. Swapping the order of the steps gives $(-1, 2)$ instead of $(-2, 1)$.

### Optional comparison: NumPy

*Optional: you can skip this cell, and nothing later depends on it.* NumPy's `@` operator and `numpy.linalg.norm` compute the same quantities as the functions above, in compiled code. If NumPy is not installed, the cell prints a note instead of failing.

In [ ]:
try:
    import numpy as np
except ImportError:
    print("NumPy is not available here; skipping this optional comparison.")
else:
    comparison_rng = random.Random(2024)
    left = [[comparison_rng.uniform(-1, 1) for _ in range(4)] for _ in range(3)]  # 3 x 4
    right = [[comparison_rng.uniform(-1, 1) for _ in range(5)] for _ in range(4)]  # 4 x 5
    ours = matrix_multiply(left, right)
    theirs = (np.array(left) @ np.array(right)).tolist()
    print("matrix_multiply agrees with NumPy's @ on a (3 x 4)(4 x 5) product:", all_close(ours, theirs))
    sample_vector = [3.0, -4.0, 12.0]
    print("Our L1, L2 and L-infinity norms:  ", norm_l1(sample_vector), norm_l2(sample_vector), norm_linf(sample_vector))
    print("NumPy's L1, L2 and L-infinity norms:", *(float(np.linalg.norm(sample_vector, order)) for order in (1, 2, np.inf)))

**Expected behaviour.** The products agree to within rounding, and both rows of norms read `19.0 13.0 12.0`.

### Optional visualisation: transforming the unit square

*Optional: you can skip this cell, and nothing later depends on it.* The plot draws the unit square and its images under a scaling, a rotation and a shear, using `transform_points` from Step 5. If Matplotlib is not installed, the cell prints a note instead of failing.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Matplotlib is not available here; skipping this optional plot.")
else:
    outline = [[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]]  # the square, closed up
    shapes = [
        ("unit square", outline, "#8a8a85", "--"),
        ("scaled by S(2, 0.5)", transform_points(scaling_matrix(2, 0.5), outline), "#2a78d6", "-"),
        ("rotated by R(30°)", transform_points(rotation_matrix(math.radians(30)), outline), "#eb6834", "-"),
        ("sheared by H(0.8)", transform_points(shear_matrix(0.8), outline), "#1baf7a", "-"),
    ]
    fig, ax = plt.subplots(figsize=(6, 4.8))
    for label, corners, colour, style in shapes:
        ax.plot([pt[0] for pt in corners], [pt[1] for pt in corners], style,
                color=colour, linewidth=2, marker="o", markersize=4, label=label)
    ax.set_aspect("equal")
    ax.grid(color="#e4e4e0", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("The unit square under three linear transformations")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2, frameon=False)
    fig.tight_layout()
    plt.show()

**Expected behaviour.** The dashed grey square is the original. The scaled square becomes a wide, flat rectangle, the rotated copy turns $30°$ anticlockwise about the origin, and the shear tilts the square into a parallelogram whose top edge has moved $0.8$ to the right.

## Exercises

Each exercise gives you a function with a docstring, a `TODO` comment and a `raise NotImplementedError(...)` line. Replace that line with your own code, run the cell, then run the `check` cell below it. Until you do, `check` prints `not implemented yet`, so the notebook still runs from top to bottom. You may use any function from the implementation steps. The reference solutions are at the end of the notebook; try each exercise before you look.

### Exercise 1: comparing vectors

* `cosine_similarity(x, y)` returns $\cos\theta = \dfrac{\mathbf{x} \cdot \mathbf{y}}{\|\mathbf{x}\|_2\, \|\mathbf{y}\|_2}$, a number between $-1$ and $1$ that compares directions and ignores lengths. It is the standard way to compare text and image embeddings. Raise `ValueError` if either vector is all zeros, as well as for the problems that `dot` already reports.
* `normalise(x)` returns the unit vector $\mathbf{x} / \|\mathbf{x}\|_2$, which points the same way as $\mathbf{x}$ and has length 1. Raise `ValueError` for an empty or all-zero vector.

In [ ]:
def cosine_similarity(x, y):
    """The cosine of the angle between x and y."""
    # TODO: reject zero vectors, then divide dot(x, y) by the product of the L2 norms.
    raise NotImplementedError("Exercise 1a: implement cosine_similarity")


def normalise(x):
    """The unit vector pointing in the same direction as x."""
    # TODO: reject empty and zero vectors, then divide every entry by norm_l2(x).
    raise NotImplementedError("Exercise 1b: implement normalise")

In [ ]:
COSINE_CASES = [
    (([1, 0], [0, 1]), 0.0),
    (([1, 2, 3], [2, 4, 6]), 1.0),
    (([1, 2], [-1, -2]), -1.0),
    (([3, 4], [4, 3]), 0.96),  # 24 / (5 * 5)
    (([0, 0], [1, 2]), ValueError),
    (([1, 2], [1, 2, 3]), ValueError),
]
NORMALISE_CASES = [
    (([3, 4],), [0.6, 0.8]),
    (([0, -2, 0],), [0.0, -1.0, 0.0]),
    (([1, 1],), [1 / math.sqrt(2), 1 / math.sqrt(2)]),
    (([0, 0],), ValueError),
    (([],), ValueError),
]
check("cosine_similarity", cosine_similarity, COSINE_CASES)
check("normalise", normalise, NORMALISE_CASES)

### Exercise 2: building and summarising matrices

* `outer_product(x, y)` returns the $m \times n$ matrix $\mathbf{x}\mathbf{y}^\top$ whose entry $(i, j)$ is $x_i\, y_j$, for vectors of lengths $m$ and $n$ (which may differ). Step 3 computed one as a column times a row; here, build it directly from the two lists. Raise `ValueError` if either vector is empty.
* `trace(A)` returns the sum of the diagonal entries, $\operatorname{tr}(A) = \sum_{i} a_{ii}$, of a **square** matrix. Raise `ValueError` for a matrix that is not square, and for empty or ragged input.

In [ ]:
def outer_product(x, y):
    """The len(x) x len(y) matrix with entries x[i] * y[j]."""
    # TODO: validate both vectors, then build one row per entry of x.
    raise NotImplementedError("Exercise 2a: implement outer_product")


def trace(A):
    """The sum of the diagonal entries of a square matrix."""
    # TODO: use shape(A) to check that A is square, then add up A[i][i].
    raise NotImplementedError("Exercise 2b: implement trace")

In [ ]:
OUTER_CASES = [
    (([1, 2], [3, 4, 5]), [[3, 4, 5], [6, 8, 10]]),
    (([2], [7]), [[14]]),
    (([1, 2, 3], [4]), [[4], [8], [12]]),
    (([0.5, -1], [2, 0]), [[1.0, 0.0], [-2, 0]]),
    (([], [1]), ValueError),
]
TRACE_CASES = [
    (([[1, 2], [3, 4]],), 5),
    (([[5]],), 5),
    (([[1, 0, 0], [0, 2, 0], [0, 0, 3]],), 6),
    (([[1, 2, 3], [4, 5, 6]],), ValueError),  # 2 x 3 is not square
    (([[1, 2], [3]],), ValueError),
    (([],), ValueError),
]
check("outer_product", outer_product, OUTER_CASES)
check("trace", trace, TRACE_CASES)

### Exercise 3: a linear layer

A fully connected (dense) layer of a neural network computes $\mathbf{y} = W\mathbf{x} + \mathbf{b}$, where $W$ is an $m \times n$ weight matrix, the input $\mathbf{x}$ has $n$ entries and the bias $\mathbf{b}$ has $m$ entries.

* `linear_layer(W, x, b)` returns $W\mathbf{x} + \mathbf{b}$ as a list of $m$ numbers. Raise `ValueError` when the sizes do not fit together.
* `batch_linear_layer(X, W, b)` applies the layer to every row of a batch $X$, an $N \times n$ matrix with one example per row, and returns the $N \times m$ matrix of outputs. Row $i$ of the result is $W\mathbf{x}_i + \mathbf{b}$; written with matrices, the result is $XW^\top$ with $\mathbf{b}$ added to every row. Validate `X` with `shape`.

In [ ]:
def linear_layer(W, x, b):
    """W x + b for an m x n matrix W, an n-vector x and an m-vector b."""
    # TODO: reuse matrix_vector_multiply and vector_add; check the length of b.
    raise NotImplementedError("Exercise 3a: implement linear_layer")


def batch_linear_layer(X, W, b):
    """Apply the layer W x + b to every row x of the batch X."""
    # TODO: validate X, then compute X W^T and add b to every row
    #       (or call your linear_layer once per row).
    raise NotImplementedError("Exercise 3b: implement batch_linear_layer")

In [ ]:
WEIGHTS = [[1, 2, 3],
           [4, 5, 6]]  # 2 x 3: three inputs, two outputs
LINEAR_CASES = [
    ((WEIGHTS, [1, 0, -1], [0.5, -0.5]), [-1.5, -2.5]),
    ((WEIGHTS, [2, 1, 0], [0, 0]), [4, 13]),
    (([[2, 0], [0, 2], [1, 1]], [3, 4], [0, 0, 1]), [6, 8, 8]),
    ((WEIGHTS, [1, 2], [0, 0]), ValueError),  # x has the wrong length
    ((WEIGHTS, [1, 2, 3], [0]), ValueError),  # b has the wrong length
]
BATCH_LINEAR_CASES = [
    (([[1, 0, -1], [2, 1, 0]], WEIGHTS, [0.5, -0.5]), [[-1.5, -2.5], [4.5, 12.5]]),
    (([[1, 1, 1]], WEIGHTS, [0, 0]), [[6, 15]]),
    (([[1, 2], [3, 4]], WEIGHTS, [0, 0]), ValueError),  # rows of length 2, but W expects 3
    (([[1, 2, 3], [4, 5]], WEIGHTS, [0, 0]), ValueError),  # a ragged batch
]
check("linear_layer", linear_layer, LINEAR_CASES)
check("batch_linear_layer", batch_linear_layer, BATCH_LINEAR_CASES)

### Exercise 4: composing transformations

* `compose_transformations(matrices)` takes a non-empty list of square matrices, all of the same size, **in the order in which they are applied**, and returns the single matrix that applies them all. For `[M1, M2, M3]` (first `M1`, then `M2`, then `M3`) the answer is $M_3 M_2 M_1$, because the rightmost matrix acts first. Raise `ValueError` for an empty list or for a matrix that is not square or not the same size as the others.
* `determinant_2x2(M)` returns $\det\begin{pmatrix} a & b \\ c & d \end{pmatrix} = ad - bc$, and raises `ValueError` unless `M` is $2 \times 2$. Its absolute value is the factor by which the transformation scales areas. A negative determinant means that the transformation also flips the plane over, and $0$ means that it squashes the plane onto a line.

The tests use the exact quarter turn `[[0, -1], [1, 0]]`, so that every expected answer is an integer you can check by hand.

In [ ]:
def compose_transformations(matrices):
    """One matrix that applies matrices[0] first, then matrices[1], and so on."""
    # TODO: validate the list, then multiply each new matrix on the LEFT of the running product.
    raise NotImplementedError("Exercise 4a: implement compose_transformations")


def determinant_2x2(M):
    """ad - bc for the 2 x 2 matrix [[a, b], [c, d]]."""
    # TODO: check that M is 2 x 2, then apply the formula.
    raise NotImplementedError("Exercise 4b: implement determinant_2x2")

In [ ]:
QUARTER_TURN = [[0, -1], [1, 0]]
DOUBLE_X = [[2, 0], [0, 1]]
SHEAR_ONE = [[1, 1], [0, 1]]
COMPOSE_CASES = [
    (([DOUBLE_X],), DOUBLE_X),
    (([DOUBLE_X, QUARTER_TURN],), [[0, -1], [2, 0]]),  # scale first, then rotate: R S
    (([QUARTER_TURN, DOUBLE_X],), [[0, -2], [1, 0]]),  # rotate first, then scale: S R
    (([QUARTER_TURN] * 4,), [[1, 0], [0, 1]]),  # four quarter turns make a full turn
    (([SHEAR_ONE] * 3,), [[1, 3], [0, 1]]),  # shears add up
    (([],), ValueError),
    (([DOUBLE_X, [[1, 2, 3], [4, 5, 6]]],), ValueError),  # not square
]
DETERMINANT_CASES = [
    (([[1, 2], [3, 4]],), -2),
    ((DOUBLE_X,), 2),  # doubling x doubles every area
    (([[2, 0], [0, 3]],), 6),
    ((QUARTER_TURN,), 1),  # rotations preserve area
    (([[1, 0.5], [0, 1]],), 1.0),  # and so do shears
    (([[1, 2], [2, 4]],), 0),  # parallel columns: the plane collapses onto a line
    (([[0, 1], [1, 0]],), -1),  # swapping x and y flips the plane over
    (([[1, 2, 3], [4, 5, 6]],), ValueError),
]
check("compose_transformations", compose_transformations, COMPOSE_CASES)
check("determinant_2x2", determinant_2x2, DETERMINANT_CASES)

## Common errors and debugging guidance

| Symptom | Likely cause | What to do |
|---|---|---|
| `[1, 2] + [3, 4]` gives `[1, 2, 3, 4]` | `+` concatenates lists | use `vector_add`, or a comprehension over `zip` |
| `2 * [1, 2]` gives `[1, 2, 1, 2]` | `*` repeats lists | use `scalar_multiply` |
| a result is silently too short | `zip` stops at the end of the shorter input | check the lengths first, or use `zip(x, y, strict=True)`, which raises `ValueError` |
| `ValueError: cannot multiply a 2x3 matrix by a 2x3 matrix` | the inner sizes differ | print `shape()` of both operands; you may need a transpose |
| the right numbers in the wrong places | rows and columns swapped, such as `A[j][i]` for `A[i][j]` | test on a non-square matrix, where a swap changes the shape |
| changing one entry changes a whole column | rows built with `[[0] * p] * m` share one list | build a new list for each row |
| a composed transformation does its steps in the wrong order | $AB$ applies $B$ first | write the steps from right to left |
| a "90 degree" rotation does something strange | `math.cos` and `math.sin` expect radians | convert with `math.radians(90)` |
| `== 0` fails after a rotation | `math.pi` is rounded, so the cosine is about `6e-17` | compare with `math.isclose(..., abs_tol=1e-12)` or `all_close` |
| `norm_l2` returns `inf` for big entries | squaring overflows | rescale the vector, or use `math.hypot` |

**Debugging habits for linear algebra**

1. **Print shapes first.** Most matrix bugs are shape bugs. Call `shape()` on every operand before you look at any numbers.
2. **Use small, non-square examples.** A $2 \times 3$ matrix exposes a swapped index immediately, while a square matrix hides it.
3. **Multiply by the identity.** $AI = A$ and $IA = A$ should hold exactly for integer matrices; if they do not, the loops are wrong.
4. **Check an identity you know.** $(AB)^\top = B^\top A^\top$ and $A(B\mathbf{x}) = (AB)\mathbf{x}$ test the indexing of a whole implementation in one line.
5. **Compute one entry by hand.** Pick entry $(i, j)$ of a failing product and work out the dot product of row $i$ and column $j$ on paper.

## Summary

* A vector is a list and a matrix is a list of equally long rows. Validate shapes, and reject empty input before you compute anything.
* Vector addition, subtraction and scalar multiplication work entry by entry; the dot product $\sum_i x_i y_i$ measures alignment, and $0$ means orthogonal.
* $(AB)_{ij} = \sum_k a_{ik} b_{kj}$: an $(m \times n)(n \times p)$ product is $m \times p$, the inner sizes must match, and the order matters.
* The transpose swaps rows and columns, $(AB)^\top = B^\top A^\top$, and the identity matrix leaves every matrix unchanged.
* The $L^1$, $L^2$ and $L^\infty$ norms measure size in different ways, and a distance is the norm of a difference.
* A $2 \times 2$ matrix transforms the plane; the product $AB$ applies $B$ first, so matrix multiplication composes transformations.

**Next:** Lesson 3 turns from the geometry of data to its uncertainty, with probability, statistics and maximum likelihood estimation built from scratch.

## Solutions

Try every exercise before you read this section. Each reference solution has a `_solution` suffix, so it never replaces your own function. Each one runs through the same `check` cases with `strict=True`, which turns `not implemented yet` into a failure, so passing here shows that the tests themselves are correct.

In [ ]:
def cosine_similarity_solution(x, y):
    length_x, length_y = norm_l2(x), norm_l2(y)  # norm_l2 rejects empty vectors
    if length_x == 0 or length_y == 0:
        raise ValueError("cosine similarity is undefined for a zero vector")
    return dot(x, y) / (length_x * length_y)  # dot rejects different lengths


def normalise_solution(x):
    length = norm_l2(x)
    if length == 0:
        raise ValueError("a zero vector has no direction")
    return [xi / length for xi in x]


check("cosine_similarity_solution", cosine_similarity_solution, COSINE_CASES, strict=True)
check("normalise_solution", normalise_solution, NORMALISE_CASES, strict=True)

In [ ]:
def outer_product_solution(x, y):
    check_vector(x, "first vector")
    check_vector(y, "second vector")
    return [[xi * yj for yj in y] for xi in x]


def trace_solution(A):
    rows, columns = shape(A)
    if rows != columns:
        raise ValueError(f"the trace needs a square matrix, got {rows} x {columns}")
    return sum(A[i][i] for i in range(rows))


check("outer_product_solution", outer_product_solution, OUTER_CASES, strict=True)
check("trace_solution", trace_solution, TRACE_CASES, strict=True)

In [ ]:
def linear_layer_solution(W, x, b):
    rows, _ = shape(W)
    if len(b) != rows:
        raise ValueError(f"the bias needs {rows} entries, got {len(b)}")
    return vector_add(matrix_vector_multiply(W, x), b)


def batch_linear_layer_solution(X, W, b):
    shape(X)  # rejects empty or ragged batches
    product = matrix_multiply(X, transpose(W))  # (N x n)(n x m) = N x m
    return [vector_add(row, b) for row in product]


check("linear_layer_solution", linear_layer_solution, LINEAR_CASES, strict=True)
check("batch_linear_layer_solution", batch_linear_layer_solution, BATCH_LINEAR_CASES, strict=True)

In [ ]:
def compose_transformations_solution(matrices):
    if len(matrices) == 0:
        raise ValueError("need at least one matrix to compose")
    size = shape(matrices[0])[0]
    for M in matrices:
        if shape(M) != (size, size):
            raise ValueError(f"every matrix must be {size} x {size}, got {shape(M)}")
    result = matrices[0]
    for M in matrices[1:]:
        result = matrix_multiply(M, result)  # the later step goes on the left
    return result


def determinant_2x2_solution(M):
    if shape(M) != (2, 2):
        raise ValueError(f"expected a 2 x 2 matrix, got shape {shape(M)}")
    (a, b), (c, d) = M
    return a * d - b * c


check("compose_transformations_solution", compose_transformations_solution, COMPOSE_CASES, strict=True)
check("determinant_2x2_solution", determinant_2x2_solution, DETERMINANT_CASES, strict=True)

**Expected behaviour.** Every solution prints a pass line, such as `compose_transformations_solution: all 7 tests passed`. If you change a solution so that it is wrong (try swapping the order of the product in `compose_transformations_solution`), its `check` call raises `AssertionError`, which shows that the tests can tell right answers from wrong ones.